# CovidGAN — reproduce the paper on Colab

Runs the full pipeline from **Waheed et al., "CovidGAN: Data Augmentation Using Auxiliary Classifier GAN for Improved Covid-19 Detection", IEEE Access 2020**: dataset assembly → train the AC-GAN → generate synthetic CXR images → train the VGG16 detection CNN with and without synthetic augmentation, and compare accuracy (paper reports 85% → 95%).

**Before running:** `Runtime > Change runtime type > T4 GPU` (or better). CovidGAN's own training loop is the slow part — the paper reports ~5h on an RTX 2060 for 2000 epochs; a Colab T4 should be in the same ballpark. Reduce `GAN_EPOCHS` below for a quick smoke test.

Repo: https://github.com/MayaHayat/CovidGAN-Pytorch

In [ ]:
!nvidia-smi

## 1. Clone the repo and install the light dependencies
Torch/torchvision are already installed (with CUDA) in the Colab image — only the extra packages need installing.

In [ ]:
!git clone https://github.com/MayaHayat/CovidGAN-Pytorch.git
%cd CovidGAN-Pytorch
!pip install -q scikit-learn matplotlib pillow

## 2. Get the datasets

**Source 1 — IEEE covid-chestxray-dataset** (COVID-CXR, public, no auth needed):

In [ ]:
!git clone --depth 1 https://github.com/ieee8023/covid-chestxray-dataset.git /content/covid-chestxray-dataset

**Source 2 — Kaggle COVID-19 Radiography Database** (Normal-CXR + more COVID-CXR).
Needs a Kaggle API token: kaggle.com → your profile → *Account* → *Create New API Token* downloads `kaggle.json`. Upload it when prompted below.

In [ ]:
from google.colab import files
print('Upload your kaggle.json:')
uploaded = files.upload()

!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d tawsifurrahman/covid19-radiography-database -p /content/kaggle_data --unzip

The Kaggle dataset's internal folder names have changed across versions — this locates the `Normal` and `COVID` class folders wherever they landed instead of hardcoding a path.

In [ ]:
from pathlib import Path

kaggle_root = Path('/content/kaggle_data')
normal_dir = next(p for p in kaggle_root.rglob('*') if p.is_dir() and p.name.lower() == 'normal')
covid_dir = next(p for p in kaggle_root.rglob('*') if p.is_dir() and p.name.lower() == 'covid')
print('Normal-CXR source:', normal_dir)
print('COVID-CXR source: ', covid_dir)

## 3. Build the train/test manifest (Sec. II-A)

Merges both sources, de-duplicates with a perceptual hash, and writes a stratified split. Defaults to the paper's own counts (331/72 COVID, 601/120 Normal train/test) when there's enough data.

In [ ]:
!python prepare_dataset.py --ieee-covid-root /content/covid-chestxray-dataset --extra-covid-dir {covid_dir} --normal-dir {normal_dir} --out-dir data --test-covid 72 --test-normal 120

## 4. Train CovidGAN — generator + discriminator (Sec. III-B)

`GAN_EPOCHS = 2000` matches the paper exactly. Drop it to a few dozen for a fast sanity check of the whole pipeline before committing to a multi-hour run.

In [ ]:
GAN_EPOCHS = 2000  # paper default; try e.g. 50 for a quick smoke test

!python train_gan.py --manifest data/manifest.csv --out-dir runs/gan --epochs {GAN_EPOCHS} --batch-size 64 --lr 2e-4 --beta1 0.5 --sample-every 25 --checkpoint-every 100

Preview the most recent sample grid (one row per class — a quick check the generator hasn't collapsed):

In [ ]:
from pathlib import Path
from IPython.display import Image, display

latest = sorted(Path('runs/gan/samples').glob('epoch_*.png'))[-1]
display(Image(filename=str(latest)))

## 5. Sample the trained generator into a synthetic augmentation pool

Defaults match the paper: 1,669 synthetic COVID-CXR, 1,399 synthetic Normal-CXR.

In [ ]:
!python generate_synthetic.py --checkpoint runs/gan/checkpoints/covidgan_final.pt --out-dir data/synthetic --n-covid 1669 --n-normal 1399

## 6. CNN-AD — detection CNN trained on real data only (baseline)

In [ ]:
!python train_classifier.py --manifest data/manifest.csv --mode ad --out-dir runs/cnn_ad --epochs 25 --batch-size 16 --lr 1e-3

## 7. CNN-SA — same CNN, trained on real + synthetic data

In [ ]:
!python train_classifier.py --manifest data/manifest.csv --mode sa --synthetic-dir data/synthetic --out-dir runs/cnn_sa --epochs 25 --batch-size 16 --lr 1e-3

## 8. Compare — this is the paper's headline result (Table 1)

In [ ]:
print('=== CNN-AD (actual data only) ===')
print(Path('runs/cnn_ad/metrics.txt').read_text())
print()
print('=== CNN-SA (actual + synthetic) ===')
print(Path('runs/cnn_sa/metrics.txt').read_text())

In [ ]:
from IPython.display import Image, display

print('CNN-AD confusion matrix')
display(Image(filename='runs/cnn_ad/confusion_matrix.png'))
print('CNN-SA confusion matrix')
display(Image(filename='runs/cnn_sa/confusion_matrix.png'))
print('PCA of penultimate-layer features (real vs. synthetic)')
display(Image(filename='runs/cnn_sa/pca.png'))

## 9. (Optional) Persist results to Google Drive

Colab sessions are ephemeral — mount Drive and copy `runs/` and `data/manifest.csv` over before the runtime recycles, especially after the multi-hour GAN training step.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/covidgan_runs
!cp -r runs /content/drive/MyDrive/covidgan_runs/
!cp data/manifest.csv /content/drive/MyDrive/covidgan_runs/